# Lab 81: The Rivalry in Your Head

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 9, *The Future Interface*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-81.ipynb)

**What you will do:** change the strength of mutual inhibition, neural fatigue and noise in the rivalry model to see how often and how regularly perception switches between the two eyes.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## Before you start

Crossing your eyes to fuse the rivalry images can cause eye strain or a mild headache in some people. If
that happens, stop and blink normally for a few seconds before trying again -- a minute or two of practice
is plenty, as the Lab's own instructions say.

## The experiment in the book

In Lab 81 you showed each eye a different image (by crossing your eyes over Figure 9.2, or with the
hole-in-the-hand method) and watched your perception alternate between the two, unable to blend them into
one stable percept. The simulation above models this as two competing neural populations whose mutual
inhibition and fatigue cause the "winner" to switch every few seconds.

## Record your results

While you have the rivalry fused (Method B), time how long each image dominates before switching, for as
many switches as you can manage comfortably. Stop immediately if your eyes strain.

In [ ]:
import pandas as pd

# example data: replace with your own timed switches, in seconds, alternating which pattern dominated
my_switches = pd.DataFrame({
    "switch_number": range(1, 7),
    "dominant": ["red", "green", "red", "green", "red", "green"],
    "duration_s": [2.1, 1.8, 2.6, 1.5, 2.9, 2.0],  # example data: replace with your own stopwatch timings
})
display(my_switches)
my_switches["duration_s"].plot.bar(color="steelblue", figsize=(6, 2.8))
plt.xlabel("switch number"); plt.ylabel("dominance duration (s)"); plt.show()
print(f"Your average dominance duration: {my_switches['duration_s'].mean():.1f} s")

## Compare

The Lab's own Simulation Box notes that "a typical run reports 5 switches over 10 seconds, an average
dominance duration of about 2 seconds", and the main text describes real rivalry alternating "every 1-3
seconds".

In [ ]:
your_mean = my_switches["duration_s"].mean()
print(f"Your average dominance duration: {your_mean:.1f} s")
if 1.0 <= your_mean <= 3.0:
    print("This falls within the 1-3 second range the Lab describes for real binocular rivalry.")
else:
    print("This falls outside the 1-3 second range in the text. Timing your own switches by eye and")
    print("stopwatch while also trying to keep your eyes crossed is a difficult dual task, so treat a")
    print("difference here as measurement noise rather than a challenge to the phenomenon.")
print()
print("For reference, the simulation above (default parameters) reports an average dominance duration of")
print("about 2.0 s over a 10 s run.")

## Simulation

You saw two images alternating even though the stimulus never changed. This simulation models binocular rivalry as two competing neural populations, each representing one eye's prediction. Mutual inhibition and adaptation cause the dominant percept to switch spontaneously.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = 0.001          # time step (seconds)
T = 10              # total time (seconds)
steps = int(T / dt)

# Parameters (after Wilson, 2003, rivalry model)
tau = 0.02              # neural time constant
beta = 1.1              # mutual inhibition strength
adaptation_rate = 0.08  # how fast an active population fatigues
recovery_rate = 0.005   # how fast fatigue recovers when suppressed
noise_level = 0.15
input_strength = 1.0

r1, r2 = 0.8, 0.2            # initial firing rates
a1, a2 = 0.0, 0.0            # initial adaptation
r1_trace, r2_trace = [], []

np.random.seed(42)
for t in range(steps):
    # Effective input = external input - adaptation
    #                    - inhibition from the other eye + noise
    eff1 = (input_strength - a1 - beta * r2
            + noise_level * np.random.randn())
    eff2 = (input_strength - a2 - beta * r1
            + noise_level * np.random.randn())

    # Firing rate update (rectified)
    r1 += dt / tau * (-r1 + max(0, eff1))
    r2 += dt / tau * (-r2 + max(0, eff2))

    # Adaptation: active neurons fatigue
    a1 += dt * (adaptation_rate * r1 - recovery_rate * a1)
    a2 += dt * (adaptation_rate * r2 - recovery_rate * a2)

    r1_trace.append(r1)
    r2_trace.append(r2)

r1_trace = np.array(r1_trace)
r2_trace = np.array(r2_trace)

time = np.linspace(0, T, steps)
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(time, r1_trace, color="red", linewidth=1.5, alpha=0.8,
        label="Left eye (red grating)")
ax.plot(time, r2_trace, color="green", linewidth=1.5, alpha=0.8,
        label="Right eye (green grating)")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Firing rate")
ax.set_title("Binocular Rivalry: Two Predictions Fighting for Dominance")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

# Smooth briefly before counting switches, so that momentary
# noise-driven crossings are not counted as separate perceptual switches.
window = 50  # 50 ms
kernel = np.ones(window) / window
smooth1 = np.convolve(r1_trace, kernel, mode="same")
smooth2 = np.convolve(r2_trace, kernel, mode="same")
switches = np.sum(np.diff(smooth1 > smooth2) != 0)
print(f"Number of perceptual switches in {T} seconds: {switches}")
print(f"Average dominance duration: {T / max(switches, 1):.1f} seconds")

**What to notice:** The red and green traces alternate dominance roughly every 1–3 seconds—exactly as you just experienced (a typical run reports 5 switches over 10 seconds, an average dominance duration of about 2 seconds). No external signal tells the brain to switch. The alternation emerges spontaneously from two competing predictions, mutual inhibition, and neural fatigue. Consciousness, in this model, is whatever prediction currently wins the competition.

## Change the parameters

**beta** sets how strongly each population suppresses the other. **adaptation_rate**
controls how quickly the currently dominant population fatigues. **noise_level** adds
randomness to each population's input. The simulation reuses the same update equations
as the cell above, run for **T** seconds.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, FloatSlider, IntSlider

def explore(beta=1.1, adaptation_rate=0.08, noise_level=0.15, T=10):
    dt = 0.001
    steps = int(T / dt)
    tau = 0.02
    recovery_rate = 0.005
    input_strength = 1.0

    r1, r2 = 0.8, 0.2
    a1, a2 = 0.0, 0.0
    r1_trace, r2_trace = [], []

    np.random.seed(42)
    for t in range(steps):
        eff1 = input_strength - a1 - beta * r2 + noise_level * np.random.randn()
        eff2 = input_strength - a2 - beta * r1 + noise_level * np.random.randn()
        r1 += dt / tau * (-r1 + max(0, eff1))
        r2 += dt / tau * (-r2 + max(0, eff2))
        a1 += dt * (adaptation_rate * r1 - recovery_rate * a1)
        a2 += dt * (adaptation_rate * r2 - recovery_rate * a2)
        r1_trace.append(r1)
        r2_trace.append(r2)

    r1_trace, r2_trace = np.array(r1_trace), np.array(r2_trace)
    time = np.linspace(0, T, steps)

    fig, ax = plt.subplots(figsize=(9, 3.2))
    ax.plot(time, r1_trace, color="red", linewidth=1.5, alpha=0.8, label="Left eye")
    ax.plot(time, r2_trace, color="green", linewidth=1.5, alpha=0.8, label="Right eye")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Firing rate")
    ax.set_title(f"beta={beta}, adaptation_rate={adaptation_rate}, noise={noise_level}")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    window = 50
    kernel = np.ones(window) / window
    smooth1 = np.convolve(r1_trace, kernel, mode="same")
    smooth2 = np.convolve(r2_trace, kernel, mode="same")
    switches = np.sum(np.diff(smooth1 > smooth2) != 0)
    print(f"Perceptual switches in {T} s: {switches}  |  Average dominance duration: {T / max(switches, 1):.1f} s")

interact(explore,
         beta=FloatSlider(value=1.1, min=0.5, max=2.0, step=0.05),
         adaptation_rate=FloatSlider(value=0.08, min=0.0, max=0.2, step=0.01),
         noise_level=FloatSlider(value=0.15, min=0.0, max=0.5, step=0.02),
         T=IntSlider(value=10, min=5, max=15, step=5));

## The AI side

The AI Connection suggests exploring "mixture-of-experts" or ensemble methods, where several models can
disagree about the same input. Below, a small neural network is trained to recognise handwritten digits,
then shown one genuinely ambiguous image -- a 50/50 blend of a "3" and an "8" -- again and again, with a
different random subset of its own units active each time (a standard technique called dropout). Like
your two eyes offering two competing interpretations of the rivalry stimulus, each "look" offers a
slightly different sub-network's opinion of the same ambiguous input.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits

torch.manual_seed(2026)
digits = load_digits()
X = digits.data / 16.0
y = digits.target
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.long)

class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 64)
        self.drop = nn.Dropout(0.5)
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        return self.fc2(self.drop(F.relu(self.fc1(x))))

net = TinyNet()
opt = torch.optim.Adam(net.parameters(), lr=0.01)
for epoch in range(80):
    opt.zero_grad()
    loss = F.cross_entropy(net(X_t), y_t)
    loss.backward()
    opt.step()
net.eval()
with torch.no_grad():
    train_acc = (net(X_t).argmax(1) == y_t).float().mean()
print(f"Training accuracy on ordinary digits: {train_acc:.0%}")

img3 = X[np.where(y == 3)[0][0]]
img8 = X[np.where(y == 8)[0][0]]
ambiguous = torch.tensor((img3 + img8) / 2, dtype=torch.float32).unsqueeze(0)

net.train()  # keep dropout switched on: each "look" samples a different sub-network
looks = 60
votes = []
with torch.no_grad():
    for _ in range(looks):
        probs = F.softmax(net(ambiguous), dim=1)[0]
        votes.append(probs.argmax().item())
votes = np.array(votes)

plt.figure(figsize=(8, 2.6))
plt.plot(votes, "o-", markersize=4, color="steelblue")
plt.yticks(sorted(set(votes)))
plt.xlabel("look number"); plt.ylabel("winning digit class")
plt.title("One ambiguous input, sampled 60 times")
plt.show()
print(pd.Series(votes).value_counts().rename("times each class won"))

If the digit "3" and "8" trade off across looks, that is the ensemble analogue of rivalry: the same
ambiguous input, two competing internal populations (here, overlapping random sub-networks), no stable
blend. The disanalogy is important: each "look" here is drawn independently at random, with no memory of
the previous look and no mutual inhibition or fatigue linking one look to the next -- unlike real rivalry
(and the simulation above), there is no reason to expect runs of several consecutive looks favouring the
same class, and no dominance-duration structure at all. It demonstrates that disagreement can arise from
a single network's own internal variability, not that this network experiences an alternating percept.

## Pool the class data

Pool dominance-duration data across the group, using anonymous IDs, to see whether the roughly 1-3
second alternation rate described in the Lab holds up across several people's eyes rather than just your
own.

In [ ]:
import io

csv_text = """id,switch_number,duration_s
P01,1,2.0
P01,2,1.7
P01,3,2.4
P02,1,1.4
P02,2,1.9
P02,3,1.6
P03,1,3.1
P03,2,2.5
P03,3,2.8
"""  # example data: replace with your class CSV
class_df = pd.read_csv(io.StringIO(csv_text))
boot = [class_df["duration_s"].sample(len(class_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean dominance duration: {class_df['duration_s'].mean():.1f} s "
      f"(95% bootstrap interval {lo:.1f}-{hi:.1f} s, n = {len(class_df)} switches)")
class_df["duration_s"].plot.hist(bins=8, color="firebrick", alpha=0.7, figsize=(6, 3))
plt.axvspan(1, 3, color="grey", alpha=0.2, label="typical range (Lab 81)")
plt.xlabel("dominance duration (s)"); plt.legend(); plt.show()

## Questions to think about

1. Lower beta below 1 (weak mutual inhibition), for example to 0.7. Does one eye come to dominate permanently, do both stay active together, or does switching continue? Compare this with what the Brain Mechanism section says rivalry requires.
2. Set adaptation_rate to 0. Does the model still switch at all? Given that adaptation is what eventually dislodges the currently dominant population, does this match what you see?
3. Raise noise_level substantially, for example to 0.4. Does the average dominance duration get shorter, longer, or more variable across settings? The lab mentions that Levelt's propositions describe an interaction between "mutual inhibition and noise-driven escape" -- which of those two ingredients does this slider control?
4. Push beta close to 2.0. Does the switching pattern still resemble the roughly 1-3 second alternation described in the lab, or does it look qualitatively different (for example, extremely rapid flickering)? What does that suggest about needing parameters within a biologically plausible range, not just any values that make the equations run?

## Challenge

Real observers differ in their average dominance duration. Explore how one parameter of
this model -- the mutual inhibition strength -- could in principle produce some of that
individual variation, by running the simulation across a range of beta values and plotting
how the average dominance duration changes.

In [ ]:
def run_rivalry(beta, T=10, seed=42):
    dt = 0.001
    steps = int(T / dt)
    tau = 0.02
    adaptation_rate, recovery_rate, noise_level, input_strength = 0.08, 0.005, 0.15, 1.0
    r1, r2, a1, a2 = 0.8, 0.2, 0.0, 0.0
    r1_trace, r2_trace = [], []
    np.random.seed(seed)
    for _ in range(steps):
        eff1 = input_strength - a1 - beta * r2 + noise_level * np.random.randn()
        eff2 = input_strength - a2 - beta * r1 + noise_level * np.random.randn()
        r1 += dt / tau * (-r1 + max(0, eff1))
        r2 += dt / tau * (-r2 + max(0, eff2))
        a1 += dt * (adaptation_rate * r1 - recovery_rate * a1)
        a2 += dt * (adaptation_rate * r2 - recovery_rate * a2)
        r1_trace.append(r1)
        r2_trace.append(r2)
    r1_trace, r2_trace = np.array(r1_trace), np.array(r2_trace)
    window = 50
    kernel = np.ones(window) / window
    smooth1 = np.convolve(r1_trace, kernel, mode="same")
    smooth2 = np.convolve(r2_trace, kernel, mode="same")
    switches = np.sum(np.diff(smooth1 > smooth2) != 0)
    return T / max(switches, 1)

betas = np.linspace(0.8, 1.8, 8)
durations = [run_rivalry(b) for b in betas]

plt.figure(figsize=(6, 3.5))
plt.plot(betas, durations, "o-", color="steelblue")
plt.xlabel("Mutual inhibition strength (beta)")
plt.ylabel("Average dominance duration (s)")
plt.title("Does inhibition strength predict dominance duration?")
plt.show()
print("This shows what the model predicts from one parameter alone; a real individual")
print("difference in dominance duration could reflect several such parameters at once.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-81.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")